# 03 – Data Warehouse & Analytics Layer
## Beauty Lakehouse Analytics Platform

This notebook implements the Data Warehouse layer of our multimodel architecture.

### Objectives
- Load curated Delta tables from the Data Lake  
- Build a Star Schema (Fact + Dimensions)  
- Prepare analytical datasets  
- Compute business KPIs:
  - Revenue per city  
  - Revenue per product category  
  - Top customers  
- Persist the analytical dataset for BI consumption  
- Generate business recommendations (OFFERS)

This layer enables optimized analytical queries for business intelligence and reporting.


In [0]:
%run ./config/settings

In [0]:
%run ./utils/load_curated_tables 


In [0]:
customers_df, products_df, orders_df, order_items_df = load_all_curated_tables(spark)


## What is a Data Warehouse?

A Data Warehouse is a structured, optimized storage layer designed for analytics.

### Compared to a Data Lake:
- Data is cleaned and structured  
- Tables follow a defined schema  
- Queries are optimized for reporting  
- Aggregations are efficient  

### Star Schema Model Used in This Notebook

#### Fact Table
- **Fact_Sales** (order_items + revenue)

#### Dimension Tables
- **Dim_Customers**  
- **Dim_Products**  
- **Dim_Orders**

This schema supports fast analytical queries and business KPI calculations.


## Building the Fact Table – Sales

The Fact table stores transactional data with measurable metrics.

### Revenue Calculation



\[
\text{Revenue} = \text{quantity} \cdot \text{unit\_price}
\]



This table will be used for all KPI calculations.


In [0]:
from pyspark.sql.functions import col 
fact_sales_df = order_items_df.withColumn( 
    "revenue", 
    col("quantity") * col("unit_price") 
) 
display(fact_sales_df.limit(5))

## Creating Dimension Tables

Dimension tables provide descriptive context for analytics.

- **Dim_Customers** → Customer information  
- **Dim_Products** → Product details  
- **Dim_Orders** → Order metadata  


In [0]:
dim_customers_df = customers_df 
dim_products_df = products_df 
dim_orders_df = orders_df 
print("Dimension tables created.")

## Joining Fact and Dimension Tables

We now join all tables to build a unified analytical dataset.  
This dataset will support KPI analysis and BI dashboards.


In [0]:
sales_analytics_df = ( 
    fact_sales_df 
    .join(dim_orders_df, "order_id") 
    .join(dim_customers_df, "customer_id") 
    .join(dim_products_df, "product_id") 
)

display(sales_analytics_df.limit(5))

## KPI 1 – Revenue per City

We calculate total revenue generated in each city to identify the strongest regional markets.


In [0]:
from pyspark.sql.functions import sum, desc

revenue_by_city_df = ( 
    sales_analytics_df 
    .groupBy("city") 
    .agg(sum("revenue")
    .alias("total_revenue")) 
    .orderBy(desc("total_revenue")) 
) 

display(revenue_by_city_df)

### Interpretation

The city with the highest total revenue represents:
- The strongest regional market  
- The most profitable sales area  
- A key target for marketing investment  

This KPI directly supports strategic decision-making.


## KPI 2 – Revenue per Product Category

We analyze which product categories generate the most revenue.


In [0]:
revenue_by_category_df = ( 
    sales_analytics_df 
    .groupBy("category") 
    .agg(sum("revenue")
    .alias("total_revenue")) 
    .orderBy(desc("total_revenue")) 
) 
    
display(revenue_by_category_df)

### Interpretation

Categories with higher revenue:
- Drive business performance  
- Should receive inventory priority  
- Are ideal candidates for targeted marketing campaigns  


## KPI 3 – Top 5 Customers by Revenue

We identify high-value customers based on total revenue.


In [0]:
top_customers_df = ( 
    sales_analytics_df 
    .groupBy("customer_id", "first_name", "last_name") 
    .agg(sum("revenue")
    .alias("total_revenue")) 
    .orderBy(desc("total_revenue")) 
    .limit(5) 
) 

display(top_customers_df)

### Interpretation

Top customers represent:
- Loyal buyers  
- High-value revenue contributors  
- Candidates for VIP or loyalty programs  


## Warehouse Schema Summary

Our analytical model follows a **Star Schema**:

### Fact Table
- **Fact_Sales**
  - order_id  
  - product_id  
  - customer_id  
  - quantity  
  - unit_price  
  - revenue  

### Dimension Tables
- **Dim_Customers**  
- **Dim_Products**  
- **Dim_Orders**  

This schema supports fast analytical queries and BI dashboards.


## Persisting Warehouse Tables

We store the analytical dataset in Parquet format.

Parquet is:
- Columnar  
- Optimized for analytics  
- Efficient for large-scale queries  


In [0]:
sales_analytics_df.write.mode(
    "overwrite").parquet(warehouse_fact_sales_path) 
print(f"Warehouse sales dataset saved to: {warehouse_fact_sales_path}")

## Business Recommendations Based on KPI Insights

The KPIs calculated in this notebook reveal clear patterns in customer behavior, product performance, and regional demand. Based on these insights, we propose the following promotional strategies:

### 1. Buy 2 items from the same category → 20% discount
Encourages customers to purchase more within a category.

### 2. City‑based discount for low-performing regions → 10% off
Targets cities with lower revenue to stimulate demand.

### 3. VIP customer loyalty discount → 15% off or free shipping
Rewards high-value customers identified in the Top Customers KPI.

### 4. Buy 2 shampoos → 20% off the second item
Shampoo is a high-volume category and suitable for bundle promotions.

### 5. Spend over 500 SEK → 50 SEK discount
Encourages larger basket sizes and increases average order value.


In [0]:
median_revenue = revenue_by_city_df.approxQuantile("total_revenue", [0.5], 0.05)[0]
top_customers = top_customers_df.select("customer_id").distinct()


In [0]:
from pyspark.sql.functions import when, col, lit 

promo_df = sales_analytics_df

promo_df = promo_df.withColumn( 
    "discount_category_20",
    when(col("quantity") >= 2, lit(0.20)).otherwise(lit(0.0)) 
)

promo_df = promo_df.join(revenue_by_city_df, "city", "left")
promo_df = promo_df.withColumn( 
    "discount_low_city_10", 
    when(col("total_revenue") < median_revenue, lit(0.10)).otherwise(lit(0.0)) 
)
promo_df = promo_df.join( 
    top_customers.withColumn("is_vip", lit(1)), 
    "customer_id", "left" )

promo_df = promo_df.withColumn(
     "discount_vip_15", when(col("is_vip") == 1, lit(0.15)).otherwise(lit(0.0)) 
)

promo_df = promo_df.withColumn( 
    "discount_shampoo_20", 
    when((col("category") == "Shampoo") & (col("quantity") >= 2), 
lit(0.20)) 
    .otherwise(lit(0.0)) 
)

promo_df = promo_df.withColumn( 
    "discount_over_500", 
    when(col("revenue") > 500, lit(50)).otherwise(lit(0)) 
)

display(promo_df.limit(20))

# Summary – Data Warehouse Layer

In this notebook we:

- Loaded curated Delta tables via a shared utility notebook  
- Built a Fact_Sales table  
- Created Dimension tables  
- Joined data into an analytical dataset  
- Calculated key business KPIs:
  - Revenue per city  
  - Revenue per category  
  - Top customers  
- Added business recommendations based on KPI insights  
- Implemented promotional rules in code  
- Persisted the analytical dataset for future BI use  

This completes the **Data Warehouse & Analytics Layer** of our Beauty Lakehouse architecture.
